<a href="https://colab.research.google.com/github/Lok-Tung/Just-for-fun/blob/main/ALM/Asset_Liability_Management_(ALM).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# 1. Data Setup
times = np.array([1, 2, 3])
liabilities = np.array([100, 110, 120])
assets_cf = np.array([[105, 0, 0], [5, 105, 0], [5, 5, 105]])
r_base = 0.05

In [3]:
# 2. Optimization
def objective_with_penalty(w, assets_cf, liabilities, times, r_base):
    optimized_cf = assets_cf.T @ w
    sse = np.sum((optimized_cf - liabilities)**2)
    df = 1 / (1 + r_base)**times
    asset_dur = np.sum(times * ((assets_cf.T @ w) * df)) / np.sum((assets_cf.T @ w) * df)
    liab_dur = np.sum(times * (liabilities * df)) / np.sum(liabilities * df)
    return sse + 100 * ((asset_dur - liab_dur)**2)

constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
result = minimize(objective_with_penalty, x0=[0.33, 0.33, 0.34],
                  args=(assets_cf, liabilities, times, r_base),
                  bounds=[(0, 1) for _ in range(3)], constraints=constraints)
optimized_w, optimized_cf = result.x, assets_cf.T @ result.x
print("Optimized Weights:", optimized_w)
print("Optimized Cash Flows:", optimized_cf)

Optimized Weights: [0.18742386 0.33145676 0.48111938]
Optimized Cash Flows: [23.74238614 37.20855631 50.51753515]


In [4]:
# 3. Metrics Calculation
df_base = 1 / (1 + r_base)**times
asset_pv = np.sum(optimized_cf * df_base)
liab_pv = np.sum(liabilities * df_base)
surplus = asset_pv - liab_pv
asset_dur = np.sum(times * (optimized_cf * df_base)) / asset_pv
liab_dur = np.sum(times * (liabilities * df_base)) / liab_pv
duration_gap = asset_dur - liab_dur

# ALM Status Logic
if abs(duration_gap) < 0.1: status = "Well Matched"
elif abs(duration_gap) < 0.5: status = "Moderate Mismatch"
else: status = "High Mismatch"

In [5]:
# 4. Scenario Analysis (bps)
scenarios = {"-200 bps": -0.02, "-100 bps": -0.01, "Base": 0, "+100 bps": 0.01, "+200 bps": 0.02}
stress_results = []
for label, shift in scenarios.items():
    df_s = 1 / (1 + (r_base + shift))**times
    stress_results.append({"Scenario": label, "Surplus": np.sum(optimized_cf * df_s) - np.sum(liabilities * df_s)})

In [6]:
# 5. Dashboard Construction
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{"type": "table"}, {"colspan": 2}, None], [{"type": "pie"}, {"colspan": 2}, None]],
    subplot_titles=("Key Risk Metrics (KPIs)", "Cash Flow Matching", "Portfolio Allocation", "Interest Rate Sensitivity Analysis")
)

# KPI Table
metrics_df = pd.DataFrame({
    "Metric": ["Asset PV", "Liability PV", "Surplus", "Asset Duration", "Liability Duration", "Duration Gap", "ALM Status"],
    "Value": [f"{asset_pv:.2f}", f"{liab_pv:.2f}", f"{surplus:.2f}", f"{asset_dur:.4f} Y", f"{liab_dur:.4f} Y", f"{duration_gap:.4f} Y", status]
})
fig.add_trace(go.Table(header=dict(values=list(metrics_df.columns)), cells=dict(values=[metrics_df.Metric, metrics_df.Value])), row=1, col=1)

# CF Matching
fig.add_trace(go.Bar(x=['Y1', 'Y2', 'Y3'], y=liabilities, name='Liability Cash Flows'), row=1, col=2)
fig.add_trace(go.Scatter(x=['Y1', 'Y2', 'Y3'], y=optimized_cf, name='Optimized Asset Cash Flows'), row=1, col=2)

# Allocation Pie
fig.add_trace(go.Pie(labels=["Short-Term Bond", "Medium-Term Bond", "Long-Term Bond"], values=optimized_w, name="Allocation"), row=2, col=1)

# Sensitivity Curve
df_stress = pd.DataFrame(stress_results)
fig.add_trace(go.Scatter(x=df_stress['Scenario'], y=df_stress['Surplus'], name='Surplus Curve', mode='lines+markers'), row=2, col=2)

fig.update_layout(height=900, width=1200, title_text="Insurance Asset-Liability Management (ALM) Dashboard")
fig.show()